# 05 · Tabela mensal

Junta as saídas das etapas 01 a 04 e o mês do ano numa tabela só, até o último mês com preço. É a entrada de todos os modelos.

In [ ]:
import sys
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, features

ETAPA = "05_tabela"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})

def ler(etapa, arquivo):
    df = pd.read_csv(config.ler_resultado(etapa, arquivo), index_col="mes")
    df.index = pd.PeriodIndex(df.index, freq="M")
    return df

## 1. Juntando

In [ ]:
preco = ler("01_preco", "preco.csv")
grupos = {
    "preco": preco.drop(columns=["preco_usd_t", "alvo"]),
    "sazonalidade": features.features_sazonalidade(preco.index),
    "enso": ler("02_enso", "oni.csv"),
    "wasde": ler("03_wasde", "wasde.csv").drop(columns="sem_relatorio"),
    "extras": ler("04_extras", "extras.csv"),
}
tabela = pd.concat([preco[["preco_usd_t"]], *grupos.values(), preco[["alvo"]]], axis=1).loc[config.INICIO:preco.index[-1]]
print(tabela.shape, "| de", tabela.index[0], "a", tabela.index[-1])

faltando = tabela.drop(columns="alvo").isna()
plt.figure(figsize=(11, 6))
plt.imshow(faltando.T, aspect="auto", cmap="Greys", interpolation="none",
           extent=[0, len(tabela), len(faltando.columns), 0])
plt.yticks(np.arange(len(faltando.columns)) + .5, faltando.columns, fontsize=8)
anos = [i for i, m in enumerate(tabela.index) if m.month == 1 and m.year % 5 == 0]
plt.xticks(anos, [tabela.index[i].year for i in anos])
plt.title("Meses sem valor (em preto)"); plt.grid(False); plt.tight_layout(); plt.show()
faltando.sum()[faltando.sum() > 0]

## 2. Features parecidas entre si

Pares com correlação acima de 0,7 trazem quase a mesma informação.

In [ ]:
corr = tabela.drop(columns=["preco_usd_t", "alvo"]).corr()
plt.figure(figsize=(10, 9))
plt.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
plt.yticks(range(len(corr)), corr.columns, fontsize=7)
plt.colorbar(shrink=.7); plt.title("Correlação entre as features"); plt.grid(False)
plt.tight_layout(); plt.savefig(PASTA / "correlacao_features.png", dpi=120); plt.show()

pares = corr.where(np.triu(np.ones(corr.shape, dtype=bool), 1)).stack()
pares[pares.abs() > .7].sort_values(key=abs, ascending=False).round(2)

## 3. Salvando

In [ ]:
saida = tabela.copy()
saida.index = saida.index.astype(str)
saida.index.name = "mes"
saida.to_csv(PASTA / "tabela_mensal.csv")

colunas_por_grupo = {nome: list(df.columns) for nome, df in grupos.items()}
config.salvar_config(PASTA, meses=len(saida), primeiro_mes=saida.index[0], ultimo_mes=saida.index[-1],
                     mes_a_prever=saida.index[-1], grupos=colunas_por_grupo)
print(sorted(p.name for p in PASTA.iterdir()))